# Bhavya — Configuration D Conformal Prediction

This notebook implements the assigned model-development work:

- validate the ClinVar dataset;
- use Configuration D with all 28 features;
- train a Gaussian generative classifier;
- reserve a gene-disjoint calibration set;
- build split conformal prediction sets;
- compare global and class-conditional calibration;
- save coverage and efficiency results.

The notebook is designed to run from the root of the `DATA_245_GroupProject` repository.


In [1]:
from pathlib import Path
import json
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.special import expit, logsumexp
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.mixture import GaussianMixture
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import (
    roc_auc_score, average_precision_score, accuracy_score,
    balanced_accuracy_score, confusion_matrix
)

warnings.filterwarnings("ignore")
SEED = 8
ALPHAS = [0.05, 0.10, 0.20]
RESULTS_DIR = Path("results")
RESULTS_DIR.mkdir(exist_ok=True)
print("Setup complete")


Setup complete


## 1. Load the shared dataset

The repository stores the processed data under `data/processed`, not `Dataset_preparation`.


In [4]:
# Locate the dataset in either the GitHub repository or Google Colab.
DATA_FILE = Path("data/processed/clinvar_expanded_model_features.csv")
if not DATA_FILE.exists():
    matches = list(Path(".").rglob("clinvar_expanded_model_features.csv"))
    if matches:
        DATA_FILE = matches[0]

if not DATA_FILE.exists():
    # In Colab, upload the CSV when it is not already present.
    try:
        from google.colab import files
        print("Upload clinvar_expanded_model_features.csv")
        uploaded = files.upload()
        csv_files = [Path(name) for name in uploaded if name.lower().endswith(".csv")]
        if csv_files:
            DATA_FILE = csv_files[0]
    except ImportError:
        pass

if not DATA_FILE.exists():
    raise FileNotFoundError(
        "Upload clinvar_expanded_model_features.csv or run this notebook from the repository root."
    )

df = pd.read_csv(DATA_FILE)

# Use the repository JSON when available. If only the CSV was uploaded to Colab,
# recreate the documented feature groups used by Configuration D.
config_matches = list(Path(".").rglob("feature_config.json"))
if config_matches:
    with open(config_matches[0]) as f:
        cfg = json.load(f)
    feature_sets = cfg["feature_sets"]
else:
    conservation = [
        "phyloP17way_primate", "phastCons17way_primate",
        "phyloP470way_mammalian", "phastCons470way_mammalian",
        "phyloP100way_vertebrate", "phastCons100way_vertebrate",
    ]
    population = [
        "gnomADe_AF", "gnomADe_AF_missing", "gnomADe_AF_neglog10",
        "gnomADg_AF", "gnomADg_AF_missing", "gnomADg_AF_neglog10",
        "gnomAD_AF", "gnomAD_AF_missing", "gnomAD_AF_neglog10",
    ]
    amino_acid = [
        "AA_hydropathy_change", "AA_hydropathy_abs_change",
        "AA_mw_change", "AA_mw_abs_change", "AA_volume_change",
        "AA_volume_abs_change", "AA_polarity_change",
        "AA_polarity_abs_change", "AA_charge_change",
        "AA_charge_abs_change", "AA_aromatic_change",
        "AA_aromatic_status_changed", "BLOSUM62_score",
    ]
    feature_sets = {
        "A": conservation,
        "B": conservation + population,
        "D": conservation + population + amino_acid,
    }
    cfg = {"target": "label", "group_column": "GeneSymbol", "feature_sets": feature_sets}

features_d = feature_sets["D"]
missing_features = [f for f in features_d if f not in df.columns]
if missing_features:
    raise ValueError(f"Configuration D features missing from the CSV: {missing_features}")

print("Dataset:", DATA_FILE)
print("Rows:", len(df))
print("Configuration D features:", len(features_d))
print(features_d)


Dataset: clinvar_expanded_model_features.csv
Rows: 46012
Configuration D features: 28
['phyloP17way_primate', 'phastCons17way_primate', 'phyloP470way_mammalian', 'phastCons470way_mammalian', 'phyloP100way_vertebrate', 'phastCons100way_vertebrate', 'gnomADe_AF', 'gnomADe_AF_missing', 'gnomADe_AF_neglog10', 'gnomADg_AF', 'gnomADg_AF_missing', 'gnomADg_AF_neglog10', 'gnomAD_AF', 'gnomAD_AF_missing', 'gnomAD_AF_neglog10', 'AA_hydropathy_change', 'AA_hydropathy_abs_change', 'AA_mw_change', 'AA_mw_abs_change', 'AA_volume_change', 'AA_volume_abs_change', 'AA_polarity_change', 'AA_polarity_abs_change', 'AA_charge_change', 'AA_charge_abs_change', 'AA_aromatic_change', 'AA_aromatic_status_changed', 'BLOSUM62_score']


## 2. Dataset validation

These checks support the data-validation and data-card deliverables.


In [5]:
assert cfg["target"] == "label"
assert cfg["group_column"] == "GeneSymbol"
assert df["VariationID"].is_unique, "VariationID must be unique"
assert df["GeneSymbol"].notna().all(), "GeneSymbol contains missing values"
assert df["label"].notna().all(), "label contains missing values"
assert set(df["label"].unique()).issubset({0, 1}), "label must be binary"
assert all(f in df.columns for f in features_d), "A Configuration D feature is missing"

missing_summary = pd.DataFrame({
    "feature": features_d,
    "missing_count": [df[f].isna().sum() for f in features_d],
    "missing_percent": [100 * df[f].isna().mean() for f in features_d],
    "unique_values": [df[f].nunique(dropna=True) for f in features_d],
    "dtype": [str(df[f].dtype) for f in features_d],
})

constant_features = missing_summary.loc[
    missing_summary["unique_values"] <= 1, "feature"
].tolist()

print("Label counts:")
print(df["label"].value_counts().sort_index())
print("Unique genes:", df["GeneSymbol"].nunique())
print("Constant features:", constant_features)
display(missing_summary.sort_values("missing_count", ascending=False))

missing_summary.to_csv(RESULTS_DIR / "bhavya_data_validation_summary.csv", index=False)


Label counts:
label
0    32013
1    13999
Name: count, dtype: int64
Unique genes: 2105
Constant features: ['gnomAD_AF_missing']


,feature,missing_count,missing_percent,unique_values,dtype
9,gnomADg_AF,12600,27.384161,6972,float64
11,gnomADg_AF_neglog10,12600,27.384161,6972,float64
8,gnomADe_AF_neglog10,1051,2.284187,11674,float64
6,gnomADe_AF,1051,2.284187,11674,float64
3,phastCons470way_mammalian,0,0.000000,947,float64
2,phyloP470way_mammalian,0,0.000000,12677,float64
1,phastCons17way_primate,0,0.000000,1001,float64
0,phyloP17way_primate,0,0.000000,2049,float64
7,gnomADe_AF_missing,0,0.000000,2,int64
5,phastCons100way_vertebrate,0,0.000000,996,float64


## 3. Gene-disjoint fit, calibration, and test split

The calibration set must be separate from model fitting. Gene-disjoint splitting prevents variants from the same gene appearing in different partitions.


In [6]:
X_raw = df[features_d].to_numpy(dtype=float)
y = df["label"].to_numpy(dtype=int)
groups = df["GeneSymbol"].to_numpy()

gss = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=SEED)
fit_cal_idx, test_idx = next(gss.split(X_raw, y, groups))

gss_cal = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=SEED)
fit_rel, cal_rel = next(
    gss_cal.split(X_raw[fit_cal_idx], y[fit_cal_idx], groups[fit_cal_idx])
)
fit_idx = fit_cal_idx[fit_rel]
cal_idx = fit_cal_idx[cal_rel]

def gene_set(idx):
    return set(groups[idx])

assert gene_set(fit_idx).isdisjoint(gene_set(cal_idx))
assert gene_set(fit_idx).isdisjoint(gene_set(test_idx))
assert gene_set(cal_idx).isdisjoint(gene_set(test_idx))

split_summary = pd.DataFrame({
    "split": ["fit", "calibration", "test"],
    "rows": [len(fit_idx), len(cal_idx), len(test_idx)],
    "genes": [len(gene_set(fit_idx)), len(gene_set(cal_idx)), len(gene_set(test_idx))],
    "pathogenic_rate": [y[fit_idx].mean(), y[cal_idx].mean(), y[test_idx].mean()],
})
display(split_summary)
split_summary.to_csv(RESULTS_DIR / "bhavya_split_summary.csv", index=False)


,split,rows,genes,pathogenic_rate
0,fit,27643,1263,0.330138
1,calibration,10646,421,0.254650
2,test,7723,421,0.279943


## 4. Preprocess using fit data only

Missing values are replaced with fit-set medians. Standardization parameters are also learned only from the fit set.


In [7]:
imputer = SimpleImputer(strategy="median")
scaler = StandardScaler()

X_fit = scaler.fit_transform(imputer.fit_transform(X_raw[fit_idx]))
X_cal = scaler.transform(imputer.transform(X_raw[cal_idx]))
X_test = scaler.transform(imputer.transform(X_raw[test_idx]))

y_fit, y_cal, y_test = y[fit_idx], y[cal_idx], y[test_idx]
print(X_fit.shape, X_cal.shape, X_test.shape)


(27643, 28) (10646, 28) (7723, 28)


## 5. Train the Configuration D Gaussian generative classifier

A separate Gaussian mixture is fit for benign and pathogenic variants. BIC selects the number of components for each class.


In [8]:
def fit_best_class_gmm(X_class, seed=SEED, max_components=5):
    candidates = []
    for covariance_type in ["diag", "tied"]:
        for k in range(1, max_components + 1):
            model = GaussianMixture(
                n_components=k,
                covariance_type=covariance_type,
                random_state=seed,
                n_init=5,
                max_iter=300,
                reg_covar=1e-3,
            )
            model.fit(X_class)
            candidates.append((model.bic(X_class), covariance_type, k, model))
    return min(candidates, key=lambda x: x[0])

models = {}
selected = []
for class_value, class_name in [(0, "benign"), (1, "pathogenic")]:
    best_bic, cov_type, k, model = fit_best_class_gmm(
        X_fit[y_fit == class_value], seed=SEED + class_value
    )
    models[class_value] = model
    selected.append({
        "class": class_name,
        "components": k,
        "covariance_type": cov_type,
        "bic": best_bic,
    })

selected_settings = pd.DataFrame(selected)
display(selected_settings)
selected_settings.to_csv(RESULTS_DIR / "bhavya_config_d_selected_gmm.csv", index=False)


,class,components,covariance_type,bic
0,benign,5,diag,539540.145537
1,pathogenic,4,diag,144698.766484


In [9]:
def log_odds_score(X):
    log_likelihood_0 = models[0].score_samples(X)
    log_likelihood_1 = models[1].score_samples(X)
    prior_1 = np.clip(y_fit.mean(), 1e-6, 1 - 1e-6)
    prior_0 = 1 - prior_1
    return (
        log_likelihood_1 + np.log(prior_1)
        - log_likelihood_0 - np.log(prior_0)
    )

score_cal = log_odds_score(X_cal)
score_test = log_odds_score(X_test)
prob_cal = expit(score_cal)
prob_test = expit(score_test)

pred_test = (prob_test >= 0.5).astype(int)
print("ROC-AUC:", roc_auc_score(y_test, score_test))
print("PR-AUC:", average_precision_score(y_test, prob_test))
print("Accuracy:", accuracy_score(y_test, pred_test))
print("Balanced accuracy:", balanced_accuracy_score(y_test, pred_test))


ROC-AUC: 0.8617294089719918
PR-AUC: 0.7128513424196177
Accuracy: 0.7952868056454746
Balanced accuracy: 0.7905641093375115


## 6. Split conformal prediction

For each example, the nonconformity score is `1 - probability of the true class`. The calibration quantile determines which labels are included in the prediction set.


In [10]:
def probability_matrix(pathogenic_probability):
    return np.column_stack([1 - pathogenic_probability, pathogenic_probability])

p_cal = probability_matrix(prob_cal)
p_test = probability_matrix(prob_test)

def conformal_quantile(scores, alpha):
    n = len(scores)
    level = min(1.0, np.ceil((n + 1) * (1 - alpha)) / n)
    return np.quantile(scores, level, method="higher")

def calibrate_global(p_cal, y_cal, alpha):
    scores = 1 - p_cal[np.arange(len(y_cal)), y_cal]
    return {"global": conformal_quantile(scores, alpha)}

def calibrate_class_conditional(p_cal, y_cal, alpha):
    thresholds = {}
    for class_value in [0, 1]:
        class_scores = 1 - p_cal[y_cal == class_value, class_value]
        thresholds[class_value] = conformal_quantile(class_scores, alpha)
    return thresholds

def make_prediction_sets(p, thresholds):
    result = np.zeros_like(p, dtype=bool)
    if "global" in thresholds:
        result = (1 - p) <= thresholds["global"]
    else:
        for class_value in [0, 1]:
            result[:, class_value] = (1 - p[:, class_value]) <= thresholds[class_value]
    return result

def evaluate_prediction_sets(prediction_sets, labels):
    contains_true = prediction_sets[np.arange(len(labels)), labels]
    sizes = prediction_sets.sum(axis=1)
    return {
        "coverage": contains_true.mean(),
        "average_set_size": sizes.mean(),
        "empty_set_rate": (sizes == 0).mean(),
        "ambiguous_set_rate": (sizes == 2).mean(),
    }


## 7. Compare calibration methods and coverage


In [11]:
rows = []
for alpha in ALPHAS:
    global_thresholds = calibrate_global(p_cal, y_cal, alpha)
    global_sets = make_prediction_sets(p_test, global_thresholds)
    metrics = evaluate_prediction_sets(global_sets, y_test)
    rows.append({
        "method": "global_split_conformal",
        "alpha": alpha,
        **metrics,
    })

    class_thresholds = calibrate_class_conditional(p_cal, y_cal, alpha)
    class_sets = make_prediction_sets(p_test, class_thresholds)
    metrics = evaluate_prediction_sets(class_sets, y_test)
    rows.append({
        "method": "class_conditional_conformal",
        "alpha": alpha,
        **metrics,
    })

comparison = pd.DataFrame(rows)
display(comparison)
comparison.to_csv(RESULTS_DIR / "bhavya_conformal_calibration_comparison.csv", index=False)


,method,alpha,coverage,average_set_size,empty_set_rate,ambiguous_set_rate
0,global_split_conformal,0.05,0.959083,1.472485,0.0,0.472485
1,class_conditional_conformal,0.05,0.964781,1.589667,0.0,0.589667
2,global_split_conformal,0.10,0.922051,1.327593,0.0,0.327593
3,class_conditional_conformal,0.10,0.925418,1.382364,0.0,0.382364
4,global_split_conformal,0.20,0.841901,1.099961,0.0,0.099961
5,class_conditional_conformal,0.20,0.839182,1.084682,0.0,0.084682


## 8. Detailed result for alpha = 0.10

The target coverage is approximately 90%. Coverage should be interpreted together with prediction-set size: a larger set is more cautious but less informative.


In [12]:
alpha = 0.10
global_thresholds = calibrate_global(p_cal, y_cal, alpha)
global_sets = make_prediction_sets(p_test, global_thresholds)
class_thresholds = calibrate_class_conditional(p_cal, y_cal, alpha)
class_sets = make_prediction_sets(p_test, class_thresholds)

detailed = pd.DataFrame([
    {"method": "global_split_conformal", **evaluate_prediction_sets(global_sets, y_test)},
    {"method": "class_conditional_conformal", **evaluate_prediction_sets(class_sets, y_test)},
])
display(detailed)


,method,coverage,average_set_size,empty_set_rate,ambiguous_set_rate
0,global_split_conformal,0.922051,1.327593,0.0,0.327593
1,class_conditional_conformal,0.925418,1.382364,0.0,0.382364


## 9. Save predictions for review


In [13]:
prediction_output = pd.DataFrame({
    "VariationID": df.iloc[test_idx]["VariationID"].to_numpy(),
    "GeneSymbol": df.iloc[test_idx]["GeneSymbol"].to_numpy(),
    "label": y_test,
    "p_pathogenic": prob_test,
    "predicted_label": pred_test,
    "global_contains_benign": global_sets[:, 0],
    "global_contains_pathogenic": global_sets[:, 1],
    "class_contains_benign": class_sets[:, 0],
    "class_contains_pathogenic": class_sets[:, 1],
})
prediction_output.to_csv(RESULTS_DIR / "bhavya_config_d_conformal_predictions.csv", index=False)
print("Saved Bhavya's Configuration D results to", RESULTS_DIR.resolve())


Saved Bhavya's Configuration D results to /content/results


In [14]:
from pathlib import Path
import shutil
import pandas as pd

RESULTS_DIR = Path("results")
RESULTS_DIR.mkdir(exist_ok=True)

# Save important tables created during the notebook
optional_tables = {
    "bhavya_conformal_calibration_comparison.csv": "comparison",
    "bhavya_config_d_selected_gmm.csv": "selected_settings",
    "bhavya_split_summary.csv": "split_summary",
    "bhavya_data_validation_summary.csv": "missing_summary",
    "bhavya_detailed_conformal_results.csv": "detailed",
    "bhavya_config_d_conformal_predictions.csv": "prediction_output",
}

for filename, variable_name in optional_tables.items():
    if variable_name in globals() and isinstance(
        globals()[variable_name], pd.DataFrame
    ):
        globals()[variable_name].to_csv(
            RESULTS_DIR / filename,
            index=False
        )

# Create a list of all saved result files
result_files = sorted(
    p for p in RESULTS_DIR.rglob("*") if p.is_file()
)

manifest = pd.DataFrame({
    "file": [
        str(p.relative_to(RESULTS_DIR))
        for p in result_files
    ],
    "size_bytes": [
        p.stat().st_size
        for p in result_files
    ],
})

manifest.to_csv(
    RESULTS_DIR / "results_manifest.csv",
    index=False
)

# Create one ZIP file containing all results
zip_path = shutil.make_archive(
    "Bhavya_Configuration_D_Conformal_Results",
    "zip",
    root_dir=str(RESULTS_DIR)
)

print(f"Saved {len(result_files)} result files.")
print(f"ZIP file created: {zip_path}")

# Automatically download the ZIP file in Google Colab
try:
    from google.colab import files
    files.download(zip_path)
except ImportError:
    print("Download the ZIP file manually from the current folder.")

Saved 6 result files.
ZIP file created: /content/Bhavya_Configuration_D_Conformal_Results.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Interpretation checklist

In the report, discuss:

1. Which Configuration D features contain missing values?
2. Whether the classes are balanced.
3. Which covariance type and number of GMM components BIC selected.
4. Whether 90% conformal coverage was achieved.
5. Which method gives better class-wise coverage.
6. The trade-off between coverage and average prediction-set size.

This notebook does not push changes to GitHub. After reviewing the results, commit the notebook and generated CSV summaries to your project branch.
